# Laboratório da Gold

O planejamento da Gold deixou três decisões para resolver com o dado na mão, antes de escrever o script de produção:

- **Decisão 1** — a taxa oficial usa média ponderada pelo `peso_aluno` ou contagem simples?
- **Decisão 4** — qual é o denominador do indicador (presentes com nota, presentes, todos)?
- **Decisão 5** — qual snapshot de meta vale, já que as metas têm linhas por ano de referência e houve revisão entre snapshots?

A régua é a mesma das outras camadas: comparar cada alternativa com o gabarito oficial (`silver/resultados/`) e só levar para o script o que bater.

In [1]:
from pathlib import Path

import pandas as pd

SILVER = Path("../data/silver")
pd.set_option("display.max_columns", 40)

alunos = pd.read_parquet(SILVER / "alunos",
                         columns=["ano", "id_municipio", "rede_nome", "proficiencia",
                                  "peso_aluno", "presente", "sem_nota"])
municipio = pd.read_parquet(SILVER / "resultados" / "municipio")
metas = pd.read_parquet(SILVER / "metas")

print(f"alunos: {len(alunos):,} | resultados municipio: {len(municipio):,} | metas: {len(metas):,}")

alunos: 3,867,589 | resultados municipio: 23,995 | metas: 10,788


## Decisão 1 — ponderada ou simples?

Recalculo a taxa de alfabetização por (ano, município) das duas formas — média simples e média ponderada pelo `peso_aluno` — sempre sobre presentes com nota, e comparo com a `taxa_alfabetizacao` oficial.

Um detalhe que importa: o gabarito municipal é **por rede** (`municipal`, `estadual`, `publica`, `total`), então o confronto honesto é rede a rede. A rede `publica` do gabarito equivale a municipal + estadual dos microdados; a `total` inclui tudo (até a privada residual).

In [2]:
com_nota = alunos[alunos["presente"] & ~alunos["sem_nota"]].copy()
com_nota["alfa"] = (com_nota["proficiencia"] >= 743).astype(float)
com_nota["alfa_peso"] = com_nota["alfa"] * com_nota["peso_aluno"]


def compara_com_gabarito(sub, rede_gabarito):
    g = sub.groupby(["ano", "id_municipio"]).agg(
        soma_alfa=("alfa", "sum"), n=("alfa", "size"),
        soma_alfa_peso=("alfa_peso", "sum"), soma_peso=("peso_aluno", "sum"),
    ).reset_index()
    g["simples"] = 100 * g["soma_alfa"] / g["n"]
    g["ponderada"] = 100 * g["soma_alfa_peso"] / g["soma_peso"]
    gab = municipio[municipio["rede_padronizada"] == rede_gabarito]
    c = g.merge(gab[["ano", "id_municipio", "taxa_alfabetizacao"]], on=["ano", "id_municipio"], how="inner")
    linhas = []
    for col in ["simples", "ponderada"]:
        d = (c[col] - c["taxa_alfabetizacao"]).abs()
        linhas.append({"rede": rede_gabarito, "taxa": col, "municipios": len(c),
                       "diff_mediana": round(d.median(), 4), "diff_p95": round(d.quantile(0.95), 4),
                       "dentro_de_0.05pp": f"{100 * (d <= 0.05).mean():.1f}%"})
    return c, pd.DataFrame(linhas)


resultados = []
for rede, filtro in [("municipal", com_nota["rede_nome"] == "municipal"),
                     ("estadual", com_nota["rede_nome"] == "estadual"),
                     ("publica", com_nota["rede_nome"].isin(["municipal", "estadual"])),
                     ("total", com_nota["rede_nome"].notna())]:
    comparacao, resumo = compara_com_gabarito(com_nota[filtro], rede)
    resultados.append(resumo)
    if rede == "publica":
        comparacao_publica = comparacao

pd.concat(resultados, ignore_index=True)

,rede,taxa,municipios,diff_mediana,diff_p95,dentro_de_0.05pp
0,municipal,simples,10269,0.2707,2.2855,23.3%
1,municipal,ponderada,10269,0.0037,0.0429,95.9%
2,estadual,simples,2139,0.1626,3.6361,41.1%
3,estadual,ponderada,2139,0.0033,0.0692,93.5%
4,publica,simples,10387,0.2810,2.3526,22.3%
5,publica,ponderada,10387,0.0040,0.0497,95.1%
6,total,simples,398,0.0025,0.0048,99.0%
7,total,ponderada,398,0.0025,0.0048,99.0%


A ponderada ganha disparado: nas redes municipal, estadual e pública, ela fecha com o gabarito em ~95% dos municípios dentro de 0,05pp, enquanto a simples só acerta ~20-40%. **Decisão 1 resolvida: a taxa oficial é a média ponderada pelo `peso_aluno`** — e a Gold calcula assim.

E o caso da rede `total` explica o que parecia estranho: ela só existe para 398 municípios, e neles simples e ponderada coincidem. São os municípios onde o peso é constante (todo aluno avaliado, sem correção amostral) — por isso lá o empate.

## Decisão 4 — qual denominador?

Três candidatos: presentes com nota, todos os presentes, todos os avaliados. Testo os três contra o gabarito da rede pública.

In [3]:
pub = alunos[alunos["rede_nome"].isin(["municipal", "estadual"])].copy()
pub["alfa"] = (pub["proficiencia"] >= 743).astype(float)
pub["alfa_peso"] = pub["alfa"] * pub["peso_aluno"]


def taxa_ponderada(sub, nome):
    g = sub.groupby(["ano", "id_municipio"]).agg(
        soma_alfa_peso=("alfa_peso", "sum"), soma_peso=("peso_aluno", "sum")).reset_index()
    g[nome] = 100 * g["soma_alfa_peso"] / g["soma_peso"]
    return g[["ano", "id_municipio", nome]]


gab_publica = municipio[municipio["rede_padronizada"] == "publica"]
teste = (taxa_ponderada(pub[pub["presente"] & ~pub["sem_nota"]], "den_com_nota")
         .merge(taxa_ponderada(pub[pub["presente"]], "den_presentes"), on=["ano", "id_municipio"])
         .merge(taxa_ponderada(pub, "den_todos"), on=["ano", "id_municipio"])
         .merge(gab_publica[["ano", "id_municipio", "taxa_alfabetizacao"]], on=["ano", "id_municipio"], how="inner"))

for col in ["den_com_nota", "den_presentes", "den_todos"]:
    d = (teste[col] - teste["taxa_alfabetizacao"]).abs()
    print(f"{col:14s} mediana={d.median():.4f}  p95={d.quantile(0.95):.4f}  dentro de 0,05pp={100 * (d <= 0.05).mean():.1f}%")

den_com_nota   mediana=0.0040  p95=0.0497  dentro de 0,05pp=95.1%
den_presentes  mediana=0.0040  p95=0.0497  dentro de 0,05pp=95.1%
den_todos      mediana=0.0040  p95=0.0497  dentro de 0,05pp=95.1%


O dado decidiu sozinho: **as três variantes dão idêntico**, porque o `peso_aluno` só existe para presentes com nota (a Silver já tinha mostrado que o nulo do peso acompanha o da proficiência). O denominador efetivo é *presentes com nota* — e a taxa de participação entra na Gold como coluna própria, calculada das contagens, sem sumir com a informação de ausência.

### Os outliers que sobram

Antes de fechar, quero olhar quem não bate: municípios com diferença acima de 1pp mesmo na ponderada.

In [4]:
comparacao_publica["diff"] = (comparacao_publica["ponderada"] - comparacao_publica["taxa_alfabetizacao"]).abs()
outliers = comparacao_publica[comparacao_publica["diff"] > 1]
print(f"outliers (> 1pp): {len(outliers)} de {len(comparacao_publica):,} ({100 * len(outliers) / len(comparacao_publica):.2f}%)")
print(outliers.groupby("ano").size().rename("municipios").to_string())
outliers.nlargest(5, "diff")[["ano", "id_municipio", "ponderada", "taxa_alfabetizacao", "n", "diff"]].round(3)

outliers (> 1pp): 45 de 10,387 (0.43%)
ano
2023    32
2024    13


,ano,id_municipio,ponderada,taxa_alfabetizacao,n,diff
922,2023,2305100,55.609,91.67,71,36.061
2932,2023,3162450,45.920,64.64,75,18.720
106,2023,1304005,57.121,40.10,121,17.021
919,2023,2304905,83.111,99.15,118,16.039
10045,2024,5103908,51.602,66.10,64,14.498


São 45 municípios (0,43%), quase todos de 2023 — o ano da Pesquisa Alfabetiza Brasil, de desenho amostral diferente. Não achei na fonte uma explicação fechada, então a decisão é pragmática: a Gold segue com o cálculo validado (que fecha com mediana de 0,004pp), o check de consistência contra o gabarito entra com tolerância, e os outliers ficam documentados no dicionário de dados em vez de travar a esteira. Se a fonte revisar, o check acusa.

## Decisão 5 — qual snapshot de meta?

As metas têm linhas por ano de referência e o planejamento registrou um caso de revisão (a meta 2024 do Brasil muda entre snapshots). Preciso saber o tamanho disso em cada nível.

In [5]:
metas_municipio = metas[metas["nivel"] == "municipio"]
piv_mun = metas_municipio.pivot_table(index="id_municipio", columns="ano",
                                      values="meta_alfabetizacao_2024", aggfunc="first").dropna()
iguais = int((piv_mun[2023] == piv_mun[2024]).sum())
print(f"municipios nos dois snapshots: {len(piv_mun):,} | meta_2024 igual: {iguais:,} | revisada: {len(piv_mun) - iguais}")

metas_uf = metas[metas["nivel"] == "uf"]
piv_uf = metas_uf.pivot_table(index="sigla_uf", columns="ano",
                              values="meta_alfabetizacao_2024", aggfunc="first")
revisadas = piv_uf[piv_uf.nunique(axis=1) > 1]
print(f"\nUFs com meta_2024 revisada entre snapshots: {len(revisadas)} de {len(piv_uf)}")
revisadas.head(8)

municipios nos dois snapshots: 5,232 | meta_2024 igual: 5,232 | revisada: 0

UFs com meta_2024 revisada entre snapshots: 22 de 24


ano,2023,2024,2025
sigla_uf,,,
AL,49.7,49.7,50.0
AM,56.8,56.8,57.0
AP,47.6,47.6,48.0
BA,43.4,43.4,43.0
ES,69.9,69.9,70.0
GO,68.9,68.9,69.0
MA,60.3,60.3,60.0
MG,63.2,63.2,63.0


O quadro ficou claro: **no nível municipal não houve revisão nenhuma** entre os snapshots 2023 e 2024. As revisões só aparecem no snapshot de 2025, nos níveis Brasil/UF — e são arredondamentos (49,7 → 50,0; 56,8 → 57,0). Como os resultados que a Gold confronta são de 2023 e 2024, a regra fica simples e sem ambiguidade:

**Decisão 5: a meta usada é a vigente no ano do resultado (linha de metas com o mesmo `ano`).** O snapshot de 2025 fica registrado, mas só entra em cena quando houver resultado de 2025 para confrontar.

## Contrato que vai para o script

O que `src/03_gold/metricas_gold.py` implementa, validado aqui:

1. **Taxa de alfabetização = média ponderada pelo `peso_aluno`** dos presentes com nota, com a regra explícita `proficiencia >= 743` (mediana de 0,004pp contra o gabarito; ~95% dos municípios dentro de 0,05pp).
2. **Denominador: presentes com nota** (o peso só existe para eles); taxa de participação como coluna própria.
3. **Meta vigente no ano do resultado** no confronto meta × resultado; 2023 sem meta pactuada (colunas começam em 2024) fica com gap nulo.
4. **Check de consistência** taxa recalculada × gabarito com tolerância, e os 45 outliers de 2023 documentados no dicionário de dados.

Os números deste notebook são os valores de referência para validar a primeira execução do script.

---

# Parte 2 — Sondagem das novas visões

A Gold de hoje responde *quanto* e *onde*: a taxa por município, o confronto com a meta, a série no tempo. Dá para ir bem além com o mesmo dado.

Aqui eu testo dez perguntas novas, uma por seção. A regra é a da Parte 1: rodar o número antes de escrever qualquer script. O que tiver sinal vira tabela; o que não tiver fica registrado como descartado, com o motivo.

| | Pergunta | Vira |
|---|---|---|
| A | Os IDs de escola e aluno são códigos reais? | limita tudo o que vem depois |
| B | A variação está entre municípios, entre escolas ou dentro da escola? | `perfil_escola` |
| C | Quais são os cortes dos 9 níveis do INEP? | `distribuicao_proficiencia` |
| D | Quantas crianças estão quase lá? | `distribuicao_proficiencia` |
| E | O indicador municipal é confiável? | `indicador_municipio` |
| F | Os ausentes puxam a taxa para cima? | `indicador_municipio` |
| G | Onde estão as crianças, em número absoluto? | `deficit_absoluto` |
| H | A rede estadual é mesmo melhor? | `recorte_desigualdade` |
| I | Quem chega em 2030? | `trajetoria_meta` |
| J | O que a camada da escola mostra? | `perfil_escola` |

## Carga

Desta vez preciso do `id_escola` e do `id_aluno` — a Parte 1 não usava nenhum dos dois.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SILVER = Path("../data/silver")
GOLD = Path("../data/gold")
CORTE = 743
pd.set_option("display.max_columns", 40)

micro = pd.read_parquet(
    SILVER / "alunos",
    columns=["ano", "id_municipio", "id_escola", "id_aluno", "sigla_uf",
             "rede_nome", "proficiencia", "peso_aluno", "presente", "sem_nota"],
)
micro["ano"] = micro["ano"].astype("int64")

publica = micro[micro["rede_nome"].isin(["municipal", "estadual"])]
com_nota = publica[publica["presente"] & ~publica["sem_nota"]].copy()
com_nota["alfa"] = (com_nota["proficiencia"] >= CORTE).astype(float)
com_nota["alfa_peso"] = com_nota["alfa"] * com_nota["peso_aluno"]

resultados_mun = pd.read_parquet(SILVER / "resultados" / "municipio")
metas = pd.read_parquet(SILVER / "metas")
indicador = pd.read_parquet(GOLD / "indicador_municipio" / "data.parquet")

# 2024 é o ano de referência da sondagem: base maior e com meta pactuada
c24 = com_nota[com_nota["ano"] == 2024].copy()
p24 = publica[publica["ano"] == 2024]


def taxa_ponderada(sub, chaves):
    """A taxa validada na Parte 1: média ponderada pelo peso_aluno."""
    g = sub.groupby(chaves, observed=True).agg(
        alunos=("alfa", "size"),
        soma_alfa_peso=("alfa_peso", "sum"),
        soma_peso=("peso_aluno", "sum"),
    )
    g["taxa"] = 100 * g["soma_alfa_peso"] / g["soma_peso"]
    return g.drop(columns=["soma_alfa_peso", "soma_peso"])


print(f"microdados: {len(micro):,} | rede pública com nota: {len(com_nota):,}")
print(f"2024: {len(c24):,} alunos, {c24['id_escola'].nunique():,} escolas, "
      f"{c24['id_municipio'].nunique():,} municípios")

## A — os IDs de escola e aluno são códigos reais?

Começo por aqui porque a resposta limita todas as outras.

Se `id_escola` for o código INEP da escola, dá para juntar com o Censo Escolar e trazer localização (urbana/rural) e infraestrutura. Seria o eixo territorial da análise de desigualdade, quase de graça.

O código INEP tem 8 dígitos e começa pelo código da UF (11 a 53). Olho os prefixos e a faixa de valores.

In [ ]:
prefixos = sorted((micro["id_escola"] // 1_000_000).unique())
print(f"prefixos de id_escola: {prefixos}")
print(f"faixa: {micro['id_escola'].min()} a {micro['id_escola'].max()}")
print(f"escolas distintas: {micro['id_escola'].nunique():,}")

# o id é estável dentro do ano?
esc_multi_mun = micro.groupby(["ano", "id_escola"])["id_municipio"].nunique()
print(f"\nescolas em mais de um município dentro do mesmo ano: {(esc_multi_mun > 1).sum()}")

# e entre anos? se for um id de verdade, o aluno repetido devia estar na mesma escola
repetidos = micro[micro.duplicated("id_aluno", keep=False)]
estab = repetidos.groupby("id_aluno").agg(muns=("id_municipio", "nunique"),
                                          escs=("id_escola", "nunique"))
print(f"id_aluno que aparece em 2023 e 2024: {len(estab):,}")
print(f"  ...no mesmo município: {100 * (estab['muns'] == 1).mean():.1f}%")
print(f"  ...na mesma escola:    {100 * (estab['escs'] == 1).mean():.1f}%")

São pseudônimos.

Os IDs vão de `60000001` a `60042811`, em sequência, todos com prefixo `60` — que não é código de UF nenhum. Conferi por fora: cruzando com as 222.589 escolas do Censo Escolar de 2023/2024, a interseção é **zero**.

E o pseudônimo muda todo ano. Dos 1,5 milhão de `id_aluno` que aparecem nos dois anos, só 0,1% caem na mesma escola. É colisão de ID, não o mesmo aluno — o que faz sentido, já que são turmas diferentes de 2º ano.

Três consequências:

- **Censo Escolar está fora.** Sem urbana/rural, sem infraestrutura da escola. O território só entra pelo município (`id_municipio` é código IBGE de verdade), e como proxy.
- **`id_aluno` não serve para juntar anos.** Quem fizer esse join vai receber lixo sem nenhum erro na tela. Vale um check de DQ para travar.
- **Mas `id_escola` serve dentro do ano.** Nenhuma escola aparece em dois municípios no mesmo ano. É chave válida para agrupar — e é o que a seção B usa.

## B — a variação está entre municípios, entre escolas ou dentro da escola?

Meta, ranking e repasse são todos pactuados no grão do município. A pergunta é se é aí que a variação está.

Quebro a variância da proficiência em três camadas: entre municípios, entre escolas do mesmo município, e entre alunos da mesma escola. Variância simples, sem peso — o peso amostral serve para calcular a taxa, aqui a leitura é estrutural.

In [ ]:
media_mun = c24.groupby("id_municipio")["proficiencia"].transform("mean")
media_esc = c24.groupby("id_escola")["proficiencia"].transform("mean")

componentes = {
    "entre municípios": media_mun.var(),
    "entre escolas (dentro do município)": (media_esc - media_mun).var(),
    "entre alunos (dentro da escola)": (c24["proficiencia"] - media_esc).var(),
}
decomp = pd.Series(componentes).to_frame("variancia")
decomp["share_%"] = (100 * decomp["variancia"] / decomp["variancia"].sum()).round(1)
print(decomp.round(1).to_string())

**O município explica 16% da variação.**

Três quartos da diferença de proficiência está entre alunos da mesma escola. Só 16% está entre municípios — justamente o recorte que a política usa para pactuar meta e cobrar resultado. A escola, que ninguém olha, responde por 9%.

Isso não invalida o indicador municipal: município é ente federado e pactua meta, escola não. Mas mostra o limite da leitura. Dois municípios com a mesma taxa podem ser muito diferentes por dentro, e a Gold não consegue mostrar isso porque para no município.

É o motivo mais forte para criar `gold/perfil_escola/`.

## C — quais são os cortes dos 9 níveis do INEP?

As tabelas `municipio` e `uf` já trazem `proporcao_aluno_nivel_0` até `_8`. É a distribuição oficial do INEP na escala do 2º ano. O problema: o dicionário da Base dos Dados não diz onde cada nível começa e termina — as colunas vêm sem a régua.

Dá para descobrir. Tenho a proporção acumulada oficial e a distribuição de proficiência dos alunos; os cortes são os quantis ponderados que correspondem a ela.

In [ ]:
niveis_oficiais = [f"proporcao_aluno_nivel_{i}" for i in range(9)]
gab24 = resultados_mun[(resultados_mun["rede_padronizada"] == "publica")
                       & (resultados_mun["ano"] == 2024)]

# proporção acumulada oficial (8 fronteiras para 9 níveis)
acumulado = gab24[niveis_oficiais].mean().cumsum().values[:-1]

ordem = np.argsort(c24["proficiencia"].values)
prof_ordenada = c24["proficiencia"].values[ordem]
peso_acumulado = 100 * np.cumsum(c24["peso_aluno"].values[ordem]) / c24["peso_aluno"].sum()

print("cortes implicados pelos quantis:",
      np.round(np.interp(acumulado, peso_acumulado, prof_ordenada), 1))

Deu `[644.8, 668.7, 692.1, 719.1, 745.2, 770.8, 795.8, 821.5]`. É uma grade de 25 em 25 a partir de 650, com o ruído esperado da aproximação (usei a média das proporções municipais no lugar da nacional).

Chute: 650, 675, 700, 725, 750, 775, 800, 825. Agora o teste de verdade — recalcular município a município e comparar com as colunas oficiais.

In [ ]:
CORTES_NIVEL = [650, 675, 700, 725, 750, 775, 800, 825]
c24["nivel"] = pd.cut(c24["proficiencia"], [-np.inf, *CORTES_NIVEL, np.inf],
                      labels=range(9), right=False)

g = c24.groupby(["id_municipio", "nivel"], observed=True)["peso_aluno"].sum().unstack(fill_value=0)
recalculado = 100 * g.div(g.sum(axis=1), axis=0)
recalculado.columns = [f"recalc_{i}" for i in recalculado.columns]

conf = recalculado.join(gab24.set_index("id_municipio")[niveis_oficiais], how="inner")
difs = pd.concat([(conf[f"recalc_{i}"] - conf[f"proporcao_aluno_nivel_{i}"]).abs()
                  for i in range(9)])

print(f"municípios conferidos: {len(conf):,} (× 9 níveis)")
print(f"mediana |diferença|: {difs.median():.4f}pp")
print(f"p95: {difs.quantile(0.95):.3f}pp | máximo: {difs.max():.2f}pp")

**Mediana de 0,003pp.** A grade está certa:

| Nível | Faixa | | Nível | Faixa |
|---|---|---|---|---|
| 0 | < 650 | | 5 | 750–775 |
| 1 | 650–675 | | 6 | 775–800 |
| 2 | 675–700 | | 7 | 800–825 |
| 3 | 700–725 | | 8 | ≥ 825 |
| 4 | 725–750 | | | |

É a mesma aderência do check da `taxa_alfabetizacao` que já existe na Gold (mediana de 0,004pp). O máximo de 9,75pp vem dos suspeitos de sempre: municípios com poucos alunos, onde uma criança muda a proporção inteira.

Duas coisas saem daqui. A `distribuicao_proficiencia` pode usar a escala oficial em vez de inventar faixa — e ganha um gabarito publicado para validar contra, que vira check permanente. E o corte de 743 cai **dentro do nível 4** (725–750): os dois sistemas não se encaixam, e isso precisa estar no dicionário de dados.

## D — quantas crianças estão quase lá?

A ideia original era faixa "Crítico < 500" e "Atenção 500–742". Antes de aceitar, confiro onde a escala começa de fato.

In [ ]:
print(f"proficiência na base: {com_nota['proficiencia'].min():.1f} a "
      f"{com_nota['proficiencia'].max():.1f}")

peso = c24["peso_aluno"]
faixas = {
    "crítico (< 700)": c24["proficiencia"] < 700,
    "atenção (700-742)": c24["proficiencia"].between(700, 742.999),
    "alfabetizado (>= 743)": c24["proficiencia"] >= CORTE,
    "--- faixa proposta: < 500": c24["proficiencia"] < 500,
}
for nome, mascara in faixas.items():
    print(f"  {nome:26s} {100 * peso[mascara].sum() / peso.sum():6.2f}%"
          f"  ({int(mascara.sum()):>9,} alunos)")

**A faixa "< 500" não pegaria ninguém.** A escala do 2º ano vai de 578 a 904. O corte de 500 vem da escala Saeb do 5º/9º ano, que é outra régua. Do jeito proposto, "Atenção" viraria sozinha todo o contingente não alfabetizado e a visão não separaria nada.

Movendo o corte para 700 (topo do nível 2, fronteira oficial), as três faixas ficam com gente nos três lados: 16,7% crítico, 24,1% atenção, 59,2% alfabetizado.

Agora a pergunta que a faixa "atenção" existe para responder: quanta criança está a poucos pontos do corte?

In [ ]:
base = 100 * (peso * (c24["proficiencia"] >= CORTE)).sum() / peso.sum()
print(f"taxa atual: {base:.2f}%\n")
for margem in [5, 10, 15, 20]:
    nova = 100 * (peso * (c24["proficiencia"] >= CORTE - margem)).sum() / peso.sum()
    print(f"  se todos a <= {margem:2d} pts do corte cruzassem: "
          f"{nova:.2f}%  (+{nova - base:.2f}pp)")

**+7,8pp se todo mundo a 10 pontos do corte cruzasse** — a taxa nacional iria de 59,2% para 67,0%. A 5 pontos, +4,1pp.

Tem um contingente enorme empilhado logo abaixo da linha. Vale uma coluna na `distribuicao_proficiencia` (uma `pct_a_10_pontos_do_corte`, por exemplo): ela responde onde o esforço rende mais, e nenhuma tabela da Gold responde isso hoje.

## E — o indicador municipal é confiável?

A `meta_vs_resultado` publica um `atingiu_meta` para cada município. Só que um município com 12 alunos avaliados tem uma taxa que balança dezenas de pontos por acaso.

Quantos desses booleanos querem dizer alguma coisa? Calculo o intervalo de confiança de 95% da taxa em cada município e comparo com o gap para a meta.

In [ ]:
mun24 = taxa_ponderada(c24, ["id_municipio"])
mun24["ic95"] = 1.96 * 100 * np.sqrt(
    (mun24["taxa"] / 100) * (1 - mun24["taxa"] / 100) / mun24["alunos"])

print(f"IC95 mediano: ±{mun24['ic95'].median():.1f}pp\n")
for lim in [30, 50, 100]:
    sub = mun24[mun24["alunos"] < lim]
    print(f"  {len(sub):>4} municípios com < {lim:3d} alunos avaliados"
          f"  | IC95 médio ±{sub['ic95'].mean():.1f}pp")

metas24 = metas[(metas["nivel"] == "municipio") & (metas["ano"] == 2024)]
metas24 = metas24[["id_municipio", "meta_alfabetizacao_2024"]].dropna().set_index("id_municipio")

confronto = mun24.join(metas24, how="inner")
confronto["gap"] = confronto["taxa"] - confronto["meta_alfabetizacao_2024"]
indistinguivel = confronto["gap"].abs() < confronto["ic95"]

print(f"\nmunicípios com meta 2024: {len(confronto):,}")
print(f"  gap menor que o próprio IC95: {indistinguivel.sum():,} "
      f"({100 * indistinguivel.mean():.1f}%)")

**43,8% dos `atingiu_meta` não são distinguíveis de zero.** Em quase metade dos municípios com meta pactuada, a diferença entre o resultado e a meta é menor que a margem de erro do próprio indicador. Nesses casos a Gold está publicando ruído com cara de fato.

E 544 municípios têm menos de 30 alunos avaliados, com margem de ±17,6pp. Sozinha, a taxa desses lugares não diz quase nada.

Isso não é tabela nova — é conserto na que já existe. `indicador_municipio` e `meta_vs_resultado` precisam publicar o `ic95` e uma coluna de confiabilidade, e o `atingiu_meta` precisa de uma versão honesta, com o estado "indistinguível". É a melhoria de melhor retorno da lista, porque corrige um problema que já está no ar.

## F — os ausentes puxam a taxa para cima?

12% dos alunos matriculados não fizeram a prova. Sem nota, eles saem do denominador: a taxa oficial é calculada só sobre quem estava lá. Se a ausência fosse aleatória, tudo bem. A pergunta é se ela é.

In [ ]:
part = p24.groupby("id_municipio").agg(avaliados=("presente", "size"),
                                       presentes=("presente", "sum"))
part["taxa_participacao"] = 100 * part["presentes"] / part["avaliados"]
part = part.join(mun24["taxa"])

ausentes = int((~p24["presente"]).sum())
print(f"ausentes em 2024: {ausentes:,} de {len(p24):,} ({100 * (~p24['presente']).mean():.1f}%)")
print(f"correlação participação × taxa: {part['taxa_participacao'].corr(part['taxa']):+.3f}")
print(f"municípios com participação < 80%: {(part['taxa_participacao'] < 80).sum()}")

# limites: e se TODOS os ausentes fossem não alfabetizados? e se todos fossem?
peso_medio = peso.mean()
soma_alfa = (peso * (c24["proficiencia"] >= CORTE)).sum()
denominador = peso.sum() + ausentes * peso_medio
limite_inf = 100 * soma_alfa / denominador
limite_sup = 100 * (soma_alfa + ausentes * peso_medio) / denominador

print(f"\ntaxa publicada: {base:.2f}%")
print(f"limites com os ausentes: [{limite_inf:.1f}% , {limite_sup:.1f}%]"
      f"  (amplitude de {limite_sup - limite_inf:.1f}pp)")

**Não é aleatória: falta mais gente onde o desempenho é pior** (correlação de +0,29 entre participação e taxa). Quem faltou tende a ser quem puxaria a média para baixo — então a taxa publicada fica inflada.

Os limites mostram o tamanho da incerteza. Se todos os ausentes fossem não alfabetizados, a taxa nacional seria 51,7%; se todos fossem alfabetizados, 64,4%. A verdade está dentro de um intervalo de 12,6pp, e o 59,2% é uma escolha metodológica dentro dele, não um fato.

Não é para trocar a taxa oficial. É para publicar a `taxa_participacao` ao lado dela e marcar os 395 municípios com participação abaixo de 80%, onde o indicador é mais frágil. Entra no mesmo pacote da seção E.

## G — onde estão as crianças, e não onde está a pior taxa

A Gold só publica percentual, e percentual esconde volume. Um município de 12 alunos com 40% aparece como "pior" que uma capital com 60% — mas a capital tem milhares de crianças fora da alfabetização e o outro tem sete.

Conto as crianças em número absoluto e vejo como o déficit se espalha pelo país.

In [ ]:
c24["nao_alfa_peso"] = (1 - c24["alfa"]) * c24["peso_aluno"]
deficit = (c24.groupby("id_municipio")
           .agg(criancas_fora=("nao_alfa_peso", "sum"), alunos=("alfa", "size"))
           .sort_values("criancas_fora", ascending=False))
deficit["acumulado_%"] = 100 * deficit["criancas_fora"].cumsum() / deficit["criancas_fora"].sum()

for p in [10, 25, 50]:
    n = int((deficit["acumulado_%"] <= p).sum()) + 1
    print(f"  {p}% das crianças não alfabetizadas estão em {n:>4} municípios "
          f"({100 * n / len(deficit):.1f}% dos municípios)")

# o ranking da taxa e o ranking do déficit são o mesmo mapa?
piores_taxa = set(mun24[mun24["alunos"] >= 50].nsmallest(50, "taxa").index)
maiores_deficit = set(deficit.head(50).index)
print(f"\nsobreposição entre 'os 50 com pior taxa' e 'os 50 com mais crianças fora': "
      f"{len(piores_taxa & maiores_deficit)} municípios")

**Metade das crianças não alfabetizadas do país está em 195 municípios** — 3,5% do total. Um quarto delas, em 34 municípios.

E o mais interessante: a sobreposição entre "os 50 com pior taxa" e "os 50 com mais crianças fora" é **zero**. Não é um mapa parecido com o outro, são dois mapas diferentes. Priorizar por taxa aponta municípios pequenos; priorizar por volume aponta as capitais. As duas leituras valem, e a Gold hoje só permite uma.

`gold/deficit_absoluto/` é barata: mesma agregação, muda o numerador.

## H — a rede estadual é mesmo melhor?

Na margem, a estadual vai melhor que a municipal. Mas as duas redes não atendem os mesmos lugares — a estadual está mais nas cidades maiores. Pode ser só efeito de contexto.

O jeito honesto de comparar é parear: olhar só os municípios onde as duas redes operam.

In [ ]:
print("na margem (sem parear):")
print(taxa_ponderada(c24, ["rede_nome"]).round(2).to_string())

pareado = taxa_ponderada(c24, ["id_municipio", "rede_nome"])["taxa"].unstack().dropna()
pareado["dif"] = pareado["estadual"] - pareado["municipal"]

print(f"\nmunicípios onde as duas redes operam: {len(pareado):,}")
print(f"diferença (estadual - municipal): média {pareado['dif'].mean():+.2f}pp"
      f" | mediana {pareado['dif'].median():+.2f}pp")
print(f"estadual vai melhor em {100 * (pareado['dif'] > 0).mean():.1f}% desses municípios")

**A vantagem estadual sobrevive ao pareamento, e ainda aumenta.** Na margem são +2,8pp. Nos 1.018 municípios onde as duas redes convivem, a mediana é **+4,25pp**, e a estadual vence em 60,3% deles.

Isso transforma um número descritivo numa conclusão defensável: não é composição, é diferença de rede. É o recorte administrativo que o projeto consegue sustentar — lembrando (seção A) que público×privado não existe nesta base e urbano×rural não é alcançável.

Entra em `recorte_desigualdade`, com a versão pareada ao lado da marginal.

## I — quem chega em 2030?

As metas municipais vão até 2030, e temos duas ondas (2023 e 2024) — o que dá um avanço observado. Comparo com o avanço que seria necessário para fechar a meta.

In [ ]:
serie = indicador.pivot_table(index="id_municipio", columns="ano",
                              values="taxa_alfabetizacao").dropna()
serie["avanco"] = serie[2024] - serie[2023]

meta2030 = metas[(metas["nivel"] == "municipio") & (metas["ano"] == 2024)]
meta2030 = (meta2030[["id_municipio", "meta_alfabetizacao_2030"]]
            .dropna().set_index("id_municipio"))

traj = serie.join(meta2030, how="inner")
traj["falta"] = traj["meta_alfabetizacao_2030"] - traj[2024]
traj["ritmo_necessario"] = traj["falta"] / 6          # de 2024 até 2030
traj["no_ritmo"] = traj["avanco"] >= traj["ritmo_necessario"]

print(f"municípios com série 2023-24 e meta 2030: {len(traj):,}\n")
print(f"avanço observado 2023->2024:  mediana {traj['avanco'].median():+.2f}pp")
print(f"ritmo necessário até 2030:   mediana {traj['ritmo_necessario'].median():+.2f}pp/ano\n")
print(f"  no ritmo:      {100 * traj['no_ritmo'].mean():.1f}%")
print(f"  fora do ritmo: {100 * (~traj['no_ritmo']).mean():.1f}%")
print(f"  já atingiram a meta 2030: {int((traj['falta'] <= 0).sum()):,}")
print(f"  pioraram entre 2023 e 2024: {int((traj['avanco'] < 0).sum()):,} "
      f"({100 * (traj['avanco'] < 0).mean():.1f}%)")

Metade dos municípios está fora do ritmo, e 42% pioraram entre 2023 e 2024. É a pergunta executiva do projeto, e o dado que já está no lake responde.

Só que tem uma armadilha aqui, e ela é séria o bastante para mudar o desenho da tabela.

In [ ]:
print(f"correlação (taxa 2023) × (avanço 2023->2024): {serie[2023].corr(serie['avanco']):+.3f}\n")
quartil = pd.qcut(serie[2023], 4, labels=["pior 25%", "2º quartil", "3º quartil", "melhor 25%"])
print("avanço mediano por quartil da taxa de 2023:")
print(serie.groupby(quartil, observed=True)["avanco"].median().round(2).to_string())

**Regressão à média.** A correlação entre a taxa de 2023 e o avanço é **−0,45**: o quartil pior "subiu" +8,2pp e o melhor "caiu" −4,3pp.

Boa parte disso não é política pública. É o município que teve azar em 2023 (poucos alunos, medida ruidosa) voltando para o próprio nível em 2024.

Ou seja: um ranking de "quem mais melhorou" seria, em boa parte, um ranking de quem teve mais ruído para baixo no ano anterior. **A Gold não pode publicar isso cru.** A `trajetoria_meta` vale a pena, mas com o IC da seção E ao lado do avanço, sem ranking de evolução, e com o aviso no dicionário de dados. Com duas ondas só, tendência é hipótese.

## J — o que a camada da escola mostra?

A seção B mostrou que a escola carrega 9% da variação, e a seção A liberou o `id_escola` para uso dentro do ano. Duas perguntas rápidas antes de fechar.

Primeira: já que o Censo Escolar está fora, o tamanho da escola serve de proxy para escola rural/multisseriada?

In [ ]:
escolas = c24.groupby("id_escola").agg(
    id_municipio=("id_municipio", "first"),
    alunos=("alfa", "size"),
    taxa=("alfa", "mean"),
    proficiencia_media=("proficiencia", "mean"),
)
escolas["taxa"] *= 100

porte = pd.cut(escolas["alunos"], [0, 10, 25, 50, 100, 100_000],
               labels=["1-10", "11-25", "26-50", "51-100", "100+"])
print(escolas.groupby(porte, observed=True).agg(
    escolas=("alunos", "size"), alunos=("alunos", "sum"),
    taxa_media=("taxa", "mean")).round(1).to_string())

O gradiente existe — 55,7% nas escolas de até 10 alunos contra 60,6% nas de 51 a 100 — mas são 5 pontos. Fraco demais para virar recorte de desigualdade, e não substitui o urbano×rural que perdemos.

Fica como feature na feature store, não como visão.

Segunda pergunta, essa mais promissora: quanto uma escola se afasta do próprio município?

In [ ]:
escolas = escolas.join(mun24["taxa"].rename("taxa_municipio"), on="id_municipio")
escolas["residuo"] = escolas["taxa"] - escolas["taxa_municipio"]

# escolas pequenas têm resíduo ruidoso pelo mesmo motivo da seção E
robustas = escolas[escolas["alunos"] >= 20]
print(f"escolas com >= 20 alunos: {len(robustas):,}")
print(f"desvio-padrão do resíduo (escola - seu município): {robustas['residuo'].std():.1f}pp\n")
print(f"  escolas 20pp ACIMA do próprio município: {int((robustas['residuo'] > 20).sum()):,}")
print(f"  escolas 20pp ABAIXO do próprio município: {int((robustas['residuo'] < -20).sum()):,}")

**2.111 escolas estão 20pp acima do próprio município.** Mesmo contexto, mesma prefeitura, mesma rede — e resultado bem melhor. Outras 2.015 estão 20pp abaixo, na situação inversa.

São essas as escolas que interessam. As de cima são casos para estudar e tentar replicar; as de baixo são onde a intervenção rende mais. A Gold hoje não consegue apontar nenhuma das duas, porque para no município.

`gold/perfil_escola/` (grão: ano + id_escola, 42 mil linhas por ano) entrega isso e a decomposição da seção B. É a tabela que mais adiciona ao projeto.

## Fechamento — o que vai para a Gold

| Seção | Resultado | Vira |
|---|---|---|
| B | 75% da variação está dentro da escola; município explica 16% | ✅ `perfil_escola` |
| C | cortes dos 9 níveis: grade de 25 pts a partir de 650 (mediana 0,003pp) | ✅ `distribuicao_proficiencia` |
| D | +7,8pp se os "a 10 pontos" cruzassem | ✅ `distribuicao_proficiencia` |
| E | 43,8% dos `atingiu_meta` não são distinguíveis de zero | ✅ conserta `indicador_municipio` e `meta_vs_resultado` |
| F | ausência não é aleatória; taxa real entre 51,7% e 64,4% | ✅ `indicador_municipio` |
| G | 50% das crianças fora em 195 municípios, sem sobreposição com o ranking de taxa | ✅ `deficit_absoluto` |
| H | estadual +4,25pp, pareado dentro do município | ✅ `recorte_desigualdade` |
| I | 49,8% fora do ritmo de 2030 | ⚠️ vale, mas **sem ranking de evolução** (regressão à média) |
| J | porte da escola como proxy de rural | ❌ 5pp de gradiente, fraco — vira feature |
| A | urbano×rural e público×privado | ❌ impossível: `id_escola` é pseudônimo, privada não existe na base |

**Se for para escolher duas:** `perfil_escola`, por causa do achado dos 75%, e o enriquecimento do `indicador_municipio` com IC e participação — esse não adiciona nada, corrige o que já está errado.

E duas coisas que a Gold não pode publicar cru, com aviso no dicionário de dados: o ranking de evolução (seção I) e o `atingiu_meta` sem o IC ao lado (seção E).